## 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


저장 장소

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/recsys_models/"
os.makedirs(SAVE_DIR, exist_ok=True)

데이터 로드

In [ ]:
import pandas as pd

train = pd.read_csv(
    "/content/drive/MyDrive/movie/ua.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "/content/drive/MyDrive/movie/ua.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

train.head()

,user_id,movie_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


timestamp 제거

In [ ]:
train = train.drop("timestamp", axis=1)
test = test.drop("timestamp", axis=1)

print(train.shape)
print(test.shape)

print(train.head())
print(test.head())

(90570, 3)
(9430, 3)
   user_id  movie_id  rating
0        1         1       5
1        1         2       3
2        1         3       4
3        1         4       3
4        1         5       3
   user_id  movie_id  rating
0        1        20       4
1        1        33       4
2        1        61       4
3        1       117       3
4        1       155       2


ranking matrix 생성

In [ ]:
train_matrix = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

train_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## train/val data 분리

In [ ]:
from sklearn.model_selection import train_test_split

train_inner, val = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["user_id"]
)

print(train_inner.shape)
print(val.shape)

(72456, 3)
(18114, 3)


# Item-Based CF

## Item-Based CF - zero fill

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

User-Item Matrix 생성

In [ ]:
train_matrix = train_inner.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

train_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1671,1672,1674,1675,1676,1677,1678,1679,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,NaN,NaN,4.0,NaN,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


결측값 0으로 채우기

In [ ]:
train_matrix_filled = train_matrix.fillna(0)

Item-Item Cosine Similarity 계산

In [ ]:
item_similarity = cosine_similarity(train_matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

item_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1671,1672,1674,1675,1676,1677,1678,1679,1681,1682
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.314152,0.246249,0.357051,0.209860,0.083805,0.455583,0.345120,0.347344,0.268149,...,0.0,0.039559,0.0,0.000000,0.000000,0.041959,0.0,0.0,0.055945,0.000000
2,0.314152,1.000000,0.169440,0.369968,0.241500,0.026413,0.278978,0.238052,0.200265,0.114372,...,0.0,0.067015,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.094774,0.094774
3,0.246249,0.169440,1.000000,0.254598,0.134046,0.075822,0.280278,0.141883,0.205756,0.141933,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,0.357051,0.369968,0.254598,1.000000,0.218684,0.063104,0.432511,0.348277,0.357961,0.233410,...,0.0,0.044682,0.0,0.105316,0.105316,0.042126,0.0,0.0,0.063189,0.084253
5,0.209860,0.241500,0.134046,0.218684,1.000000,0.019818,0.248801,0.185328,0.163054,0.063803,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.106668


Item-Based CF 예측 함수

In [ ]:
def predict_item_based(user_id, movie_id, train_matrix, item_similarity_df):
    # user가 train에 없는 경우
    if user_id not in train_matrix.index:
        return train_matrix.stack().mean()

    # movie가 train similarity에 없는 경우
    if movie_id not in item_similarity_df.index:
        return train_matrix.loc[user_id].mean()

    # 해당 사용자의 평점 벡터
    user_ratings = train_matrix.loc[user_id].dropna()

    # 사용자가 아무 영화도 평가하지 않은 경우
    if len(user_ratings) == 0:
        return train_matrix.stack().mean()

    # 사용자가 평가한 영화들과 target movie의 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    # 가중합 계산
    numerator = np.dot(similarities, user_ratings)
    denominator = np.sum(np.abs(similarities))

    if denominator == 0:
        return user_ratings.mean()

    pred = numerator / denominator

    # 평점 범위 제한
    pred = np.clip(pred, 1, 5)

    return pred

Validation 데이터 예측

In [ ]:
val_preds = []

for row in val.itertuples():
    pred = predict_item_based(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_df
    )
    val_preds.append(pred)

val["item_cf_pred"] = val_preds
val.head()

,user_id,movie_id,rating,item_cf_pred
30016,311,89,5,3.922228
51012,496,485,3,2.998037
79374,826,678,4,3.884997
30415,312,611,5,4.448797
69284,699,95,3,3.103516


RMSE 계산



In [ ]:
rmse_basic = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_pred"])
)

print("Basic Item-Based CF Validation RMSE:", rmse_basic)

Basic Item-Based CF Validation RMSE: 1.0105797462291823


## Item-Based CF - Mean-Centered

mean centering

In [ ]:
user_means = train_matrix.mean(axis=1)

In [ ]:
train_centered = train_matrix.sub(user_means, axis=0)
train_centered_filled = train_centered.fillna(0)

아이템 유사도 계산 - 0 으로 채운 값으로

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity_centered = cosine_similarity(train_centered_filled.T)

item_similarity_centered_df = pd.DataFrame(
    item_similarity_centered,
    index=train_centered.columns,
    columns=train_centered.columns
)

예측 함수

In [ ]:
import numpy as np

global_mean = train_matrix.stack().mean()

def predict_item_based_centered(user_id, movie_id, train_matrix, item_similarity_df, user_means):
    # cold-start user
    if user_id not in train_matrix.index:
        return global_mean

    # 기준 사용자 평균
    user_mean = user_means.loc[user_id]

    # cold-start item
    if movie_id not in item_similarity_df.index:
        return np.clip(user_mean, 1, 5)

    # 사용자가 평가한 영화들
    user_ratings = train_matrix.loc[user_id].dropna()

    if len(user_ratings) == 0:
        return global_mean

    # mean-centered rating
    centered_ratings = user_ratings - user_mean

    # target item과 사용자가 평가한 item들의 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    numerator = np.dot(similarities, centered_ratings)
    denominator = np.sum(np.abs(similarities))

    if denominator == 0:
        return np.clip(user_mean, 1, 5)

    pred = user_mean + numerator / denominator

    return np.clip(pred, 1, 5)

validation 예측

In [ ]:
val_centered_preds = []

for row in val.itertuples():
    pred = predict_item_based_centered(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_centered_df,
        user_means
    )
    val_centered_preds.append(pred)

val["item_cf_centered_pred"] = val_centered_preds

RMSE 계산

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_centered = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_centered_pred"])
)

print("Mean-Centered Item-Based CF Validation RMSE:", rmse_centered)

Mean-Centered Item-Based CF Validation RMSE: 0.9473802020437072


RMSE 비교

In [ ]:
print("Basic Item-Based CF RMSE:", rmse_basic)
print("Mean-Centered Item-Based CF RMSE:", rmse_centered)

Basic Item-Based CF RMSE: 1.0105797462291823
Mean-Centered Item-Based CF RMSE: 0.9473802020437072


## Item-Based CF - Mean-Centered + top-k

예측 함수

In [ ]:
global_mean = train_matrix.stack().mean()

In [ ]:
def predict_item_based_centered_topk(
    user_id,
    movie_id,
    train_matrix,
    item_similarity_df,
    user_means,
    K=30
):
    # cold-start user
    if user_id not in train_matrix.index:
        return global_mean

    user_mean = user_means.loc[user_id]

    # cold-start item
    if movie_id not in item_similarity_df.index:
        return np.clip(user_mean, 1, 5)

    # 사용자가 평가한 영화들
    user_ratings = train_matrix.loc[user_id].dropna()

    if len(user_ratings) == 0:
        return global_mean

    # target movie와 사용자가 평가한 영화들 간 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    # 자기 자신 제외
    similarities = similarities.drop(labels=[movie_id], errors="ignore")
    user_ratings = user_ratings.loc[similarities.index]

    # 유사도 높은 Top-K만 선택
    top_k_similarities = similarities.sort_values(
        ascending=False
    ).head(K)

    top_k_ratings = user_ratings.loc[top_k_similarities.index]

    if len(top_k_similarities) == 0:
        return np.clip(user_mean, 1, 5)

    # mean-centered rating
    centered_ratings = top_k_ratings - user_mean

    numerator = np.dot(top_k_similarities, centered_ratings)
    denominator = np.sum(np.abs(top_k_similarities))

    if denominator == 0:
        return np.clip(user_mean, 1, 5)

    pred = user_mean + numerator / denominator

    return np.clip(pred, 1, 5)

Top-K 별 RMSE 비교

In [ ]:
K_list = [5, 10, 20, 30, 40, 50, 100]

results = []

for K in K_list:
    preds = []

    for row in val.itertuples():
        pred = predict_item_based_centered_topk(
            row.user_id,
            row.movie_id,
            train_matrix,
            item_similarity_centered_df,
            user_means,
            K=K
        )
        preds.append(pred)

    rmse = np.sqrt(mean_squared_error(val["rating"], preds))

    results.append({
        "model": f"Mean-Centered Item-CF Top-{K}",
        "K": K,
        "RMSE": rmse
    })

results_df = pd.DataFrame(results)
results_df

,model,K,RMSE
0,Mean-Centered Item-CF Top-5,5,0.972080
1,Mean-Centered Item-CF Top-10,10,0.940072
2,Mean-Centered Item-CF Top-20,20,0.931331
3,Mean-Centered Item-CF Top-30,30,0.930488
4,Mean-Centered Item-CF Top-40,40,0.932327
5,Mean-Centered Item-CF Top-50,50,0.934035
6,Mean-Centered Item-CF Top-100,100,0.940788


In [ ]:
# Basic Item-CF RMSE
rmse_basic = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_pred"])
)

# Mean-Centered Item-CF RMSE
rmse_centered = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_centered_pred"])
)

# Top-K 중 최고 성능
best_topk = results_df.loc[results_df["RMSE"].idxmin()]

comparison_df = pd.DataFrame([
    {
        "model": "Basic Item-CF",
        "RMSE": rmse_basic
    },
    {
        "model": "Mean-Centered Item-CF",
        "RMSE": rmse_centered
    },
    {
        "model": best_topk["model"],
        "RMSE": best_topk["RMSE"]
    }
])

comparison_df

,model,RMSE
0,Basic Item-CF,1.010580
1,Mean-Centered Item-CF,0.947380
2,Mean-Centered Item-CF Top-30,0.930488


validation 예측

In [ ]:
best_k = 30

topk_preds = []

for row in val.itertuples():
    pred = predict_item_based_centered_topk(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_centered_df,
        user_means,
        K=best_k
    )

    topk_preds.append(pred)

val["item_cf_topk_pred"] = topk_preds

In [ ]:
val[[
    "rating",
    "item_cf_topk_pred"
]].head()

,rating,item_cf_topk_pred
30016,5,4.200658
51012,3,3.144088
79374,4,3.651598
30415,5,4.601417
69284,3,3.205588


# SVD

## basic SVD

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
class BasicSVD:
    def __init__(
        self,
        n_factors=50,
        n_epochs=20,
        lr=0.005,
        reg=0.02,
        random_state=1234
    ):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, train_df):
        np.random.seed(self.random_state)

        self.global_mean = train_df["rating"].mean()

        self.user_ids = train_df["user_id"].unique()
        self.item_ids = train_df["movie_id"].unique()

        self.user_to_idx = {
            user_id: idx for idx, user_id in enumerate(self.user_ids)
        }
        self.item_to_idx = {
            item_id: idx for idx, item_id in enumerate(self.item_ids)
        }

        n_users = len(self.user_ids)
        n_items = len(self.item_ids)

        self.P = np.random.normal(
            scale=0.1,
            size=(n_users, self.n_factors)
        )
        self.Q = np.random.normal(
            scale=0.1,
            size=(n_items, self.n_factors)
        )

        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)

        train_array = train_df[[
            "user_id",
            "movie_id",
            "rating"
        ]].values

        self.train_rmse_history = []

        for epoch in range(self.n_epochs):
            np.random.shuffle(train_array)

            squared_errors = []

            for user_id, item_id, rating in train_array:
                u = self.user_to_idx[user_id]
                i = self.item_to_idx[item_id]

                pred = (
                    self.global_mean
                    + self.bu[u]
                    + self.bi[i]
                    + np.dot(self.P[u], self.Q[i])
                )

                err = rating - pred
                squared_errors.append(err ** 2)

                # 기존 latent vector 저장
                p_u = self.P[u].copy()
                q_i = self.Q[i].copy()

                # bias 업데이트
                self.bu[u] += self.lr * (
                    err - self.reg * self.bu[u]
                )
                self.bi[i] += self.lr * (
                    err - self.reg * self.bi[i]
                )

                # latent factor 업데이트
                self.P[u] += self.lr * (
                    err * q_i - self.reg * p_u
                )
                self.Q[i] += self.lr * (
                    err * p_u - self.reg * q_i
                )

            train_rmse = np.sqrt(np.mean(squared_errors))
            self.train_rmse_history.append(train_rmse)

            print(
                f"Epoch {epoch + 1}/{self.n_epochs} - Train RMSE: {train_rmse:.4f}"
            )

    def predict_one(self, user_id, item_id):
        pred = self.global_mean

        if user_id in self.user_to_idx:
            u = self.user_to_idx[user_id]
            pred += self.bu[u]
        else:
            u = None

        if item_id in self.item_to_idx:
            i = self.item_to_idx[item_id]
            pred += self.bi[i]
        else:
            i = None

        if u is not None and i is not None:
            pred += np.dot(self.P[u], self.Q[i])

        return np.clip(pred, 1, 5)

    def predict(self, df):
        preds = []

        for row in df.itertuples():
            pred = self.predict_one(
                row.user_id,
                row.movie_id
            )
            preds.append(pred)

        return np.array(preds)

In [ ]:
basic_svd = BasicSVD(
    n_factors=50,
    n_epochs=20,
    lr=0.005,
    reg=0.02,
    random_state=1234
)

basic_svd.fit(train_inner)

Epoch 1/20 - Train RMSE: 1.0521
Epoch 2/20 - Train RMSE: 0.9829
Epoch 3/20 - Train RMSE: 0.9564
Epoch 4/20 - Train RMSE: 0.9405
Epoch 5/20 - Train RMSE: 0.9288
Epoch 6/20 - Train RMSE: 0.9194
Epoch 7/20 - Train RMSE: 0.9111
Epoch 8/20 - Train RMSE: 0.9035
Epoch 9/20 - Train RMSE: 0.8962
Epoch 10/20 - Train RMSE: 0.8888
Epoch 11/20 - Train RMSE: 0.8814
Epoch 12/20 - Train RMSE: 0.8736
Epoch 13/20 - Train RMSE: 0.8654
Epoch 14/20 - Train RMSE: 0.8566
Epoch 15/20 - Train RMSE: 0.8473
Epoch 16/20 - Train RMSE: 0.8373
Epoch 17/20 - Train RMSE: 0.8267
Epoch 18/20 - Train RMSE: 0.8156
Epoch 19/20 - Train RMSE: 0.8039
Epoch 20/20 - Train RMSE: 0.7920


In [ ]:
val["basic_svd_manual_pred"] = basic_svd.predict(val)

rmse_manual_svd = np.sqrt(
    mean_squared_error(
        val["rating"],
        val["basic_svd_manual_pred"]
    )
)

mae_manual_svd = mean_absolute_error(
    val["rating"],
    val["basic_svd_manual_pred"]
)

print("Manual Basic SVD RMSE:", rmse_manual_svd)
print("Manual Basic SVD MAE:", mae_manual_svd)

Manual Basic SVD RMSE: 0.9326816379401419
Manual Basic SVD MAE: 0.7351492074946665


## SVD - implicit feedback

In [ ]:
class ManualSVDpp:
    def __init__(
        self,
        n_factors=50,
        n_epochs=20,
        lr=0.005,
        reg=0.02,
        random_state=1234
    ):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, train_df):
        np.random.seed(self.random_state)

        self.global_mean = train_df["rating"].mean()

        self.user_ids = train_df["user_id"].unique()
        self.item_ids = train_df["movie_id"].unique()

        self.user_to_idx = {
            user_id: idx for idx, user_id in enumerate(self.user_ids)
        }

        self.item_to_idx = {
            item_id: idx for idx, item_id in enumerate(self.item_ids)
        }

        n_users = len(self.user_ids)
        n_items = len(self.item_ids)

        self.P = np.random.normal(
            scale=0.1,
            size=(n_users, self.n_factors)
        )

        self.Q = np.random.normal(
            scale=0.1,
            size=(n_items, self.n_factors)
        )

        # implicit feedback item latent vector
        self.Y = np.random.normal(
            scale=0.1,
            size=(n_items, self.n_factors)
        )

        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)

        # 사용자별 interaction item set N(u)
        self.user_items = {}

        for user_id, group in train_df.groupby("user_id"):
            u = self.user_to_idx[user_id]
            item_indices = [
                self.item_to_idx[item_id]
                for item_id in group["movie_id"]
                if item_id in self.item_to_idx
            ]
            self.user_items[u] = item_indices

        train_array = train_df[[
            "user_id",
            "movie_id",
            "rating"
        ]].values

        self.train_rmse_history = []

        for epoch in range(self.n_epochs):
            np.random.shuffle(train_array)

            squared_errors = []

            for user_id, item_id, rating in train_array:
                u = self.user_to_idx[user_id]
                i = self.item_to_idx[item_id]

                Nu = self.user_items[u]
                sqrt_Nu = np.sqrt(len(Nu))

                if sqrt_Nu > 0:
                    implicit_sum = np.sum(
                        self.Y[Nu],
                        axis=0
                    ) / sqrt_Nu
                else:
                    implicit_sum = np.zeros(self.n_factors)

                user_vector = self.P[u] + implicit_sum

                pred = (
                    self.global_mean
                    + self.bu[u]
                    + self.bi[i]
                    + np.dot(self.Q[i], user_vector)
                )

                err = rating - pred
                squared_errors.append(err ** 2)

                # 기존 값 저장
                p_u_old = self.P[u].copy()
                q_i_old = self.Q[i].copy()

                # bias 업데이트
                self.bu[u] += self.lr * (
                    err - self.reg * self.bu[u]
                )

                self.bi[i] += self.lr * (
                    err - self.reg * self.bi[i]
                )

                # latent vector 업데이트
                self.P[u] += self.lr * (
                    err * q_i_old - self.reg * p_u_old
                )

                self.Q[i] += self.lr * (
                    err * user_vector - self.reg * q_i_old
                )

                # implicit factor y_j 업데이트
                for j in Nu:
                    self.Y[j] += self.lr * (
                        err * q_i_old / sqrt_Nu
                        - self.reg * self.Y[j]
                    )

            train_rmse = np.sqrt(np.mean(squared_errors))
            self.train_rmse_history.append(train_rmse)

            print(
                f"Epoch {epoch + 1}/{self.n_epochs} - Train RMSE: {train_rmse:.4f}"
            )

    def predict_one(self, user_id, item_id):
        pred = self.global_mean

        if user_id in self.user_to_idx:
            u = self.user_to_idx[user_id]
            pred += self.bu[u]
        else:
            u = None

        if item_id in self.item_to_idx:
            i = self.item_to_idx[item_id]
            pred += self.bi[i]
        else:
            i = None

        if u is not None and i is not None:
            Nu = self.user_items.get(u, [])
            sqrt_Nu = np.sqrt(len(Nu))

            if sqrt_Nu > 0:
                implicit_sum = np.sum(
                    self.Y[Nu],
                    axis=0
                ) / sqrt_Nu
            else:
                implicit_sum = np.zeros(self.n_factors)

            user_vector = self.P[u] + implicit_sum

            pred += np.dot(
                self.Q[i],
                user_vector
            )

        return np.clip(pred, 1, 5)

    def predict(self, df):
        preds = []

        for row in df.itertuples():
            pred = self.predict_one(
                row.user_id,
                row.movie_id
            )
            preds.append(pred)

        return np.array(preds)

In [ ]:
manual_svdpp = ManualSVDpp(
    n_factors=20,
    n_epochs=10,
    lr=0.005,
    reg=0.02,
    random_state=1234
)

manual_svdpp.fit(train_inner)

Epoch 1/10 - Train RMSE: 1.0499
Epoch 2/10 - Train RMSE: 0.9809
Epoch 3/10 - Train RMSE: 0.9560
Epoch 4/10 - Train RMSE: 0.9416
Epoch 5/10 - Train RMSE: 0.9318
Epoch 6/10 - Train RMSE: 0.9241
Epoch 7/10 - Train RMSE: 0.9173
Epoch 8/10 - Train RMSE: 0.9112
Epoch 9/10 - Train RMSE: 0.9053
Epoch 10/10 - Train RMSE: 0.8995


In [ ]:
val["manual_svdpp_pred"] = manual_svdpp.predict(val)

rmse_manual_svdpp = np.sqrt(
    mean_squared_error(
        val["rating"],
        val["manual_svdpp_pred"]
    )
)

mae_manual_svdpp = mean_absolute_error(
    val["rating"],
    val["manual_svdpp_pred"]
)

print("Manual SVD++ RMSE:", rmse_manual_svdpp)
print("Manual SVD++ MAE:", mae_manual_svdpp)

Manual SVD++ RMSE: 0.9325269370980428
Manual SVD++ MAE: 0.7364513919121872


## Baseline + Latent Factor + Item-Item Neighborhood

In [ ]:
import numpy as np
import pandas as pd
import math
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
def clip_rating(pred):
    return np.clip(pred, 1, 5)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

In [ ]:
class IntegratedHybridManual:
    """
    Integrated Hybrid Model:
    Baseline Bias + SVD++ Latent Factor + Item-Item Neighborhood

    예측식:
    r_hat_ui =
        mu + b_u + b_i
        + q_i · (p_u + |N(u)|^-1/2 sum y_j)
        + |Q_i(u)|^-1/2 sum [ w_ji * (r_uj - b_uj) + c_ji ]
    """

    def __init__(
        self,
        n_factors=20,
        k_neighbor=30,
        n_epochs=10,
        lr=0.005,
        reg=0.02,
        random_state=1234
    ):
        self.n_factors = n_factors
        self.k_neighbor = k_neighbor
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def _make_index(self, train_df):
        self.user_ids = train_df["user_id"].unique()
        self.item_ids = train_df["movie_id"].unique()

        self.user_to_idx = {
            user_id: idx for idx, user_id in enumerate(self.user_ids)
        }
        self.item_to_idx = {
            item_id: idx for idx, item_id in enumerate(self.item_ids)
        }

        self.idx_to_user = {
            idx: user_id for user_id, idx in self.user_to_idx.items()
        }
        self.idx_to_item = {
            idx: item_id for item_id, idx in self.item_to_idx.items()
        }

        self.n_users = len(self.user_ids)
        self.n_items = len(self.item_ids)

    def _convert_df(self, df):
        temp = df.copy()

        temp["user_idx"] = temp["user_id"].map(self.user_to_idx)
        temp["item_idx"] = temp["movie_id"].map(self.item_to_idx)

        temp = temp.dropna(subset=["user_idx", "item_idx"])
        temp["user_idx"] = temp["user_idx"].astype(int)
        temp["item_idx"] = temp["item_idx"].astype(int)

        return temp

    def _build_user_history(self, train_idx_df):
        self.user_history = [dict() for _ in range(self.n_users)]

        for row in train_idx_df.itertuples():
            self.user_history[row.user_idx][row.item_idx] = float(row.rating)

        self.user_items = [
            list(history.keys()) for history in self.user_history
        ]

    def _build_item_neighbors(self, train_idx_df):
        rows = train_idx_df["user_idx"].values
        cols = train_idx_df["item_idx"].values

        # baseline 제거 전 단순 centered rating으로 item similarity 계산
        ratings = train_idx_df["rating"].values - self.mu

        R_sparse = csr_matrix(
            (ratings, (rows, cols)),
            shape=(self.n_users, self.n_items)
        )

        item_norms = np.sqrt(
            np.array(R_sparse.power(2).sum(axis=0)).flatten()
        ) + 1e-9

        sim = (R_sparse.T @ R_sparse).toarray()
        sim = sim / np.outer(item_norms, item_norms)

        np.fill_diagonal(sim, -np.inf)

        top_k = min(self.k_neighbor, self.n_items - 1)

        self.neighbors = np.argpartition(
            -sim,
            top_k,
            axis=1
        )[:, :top_k]

    def fit(self, train_df, val_df=None):
        rng = np.random.default_rng(self.random_state)

        self._make_index(train_df)

        train_idx_df = self._convert_df(train_df)

        self.mu = train_idx_df["rating"].mean()

        self.bu = np.zeros(self.n_users)
        self.bi = np.zeros(self.n_items)

        self.P = 0.1 * rng.normal(
            size=(self.n_users, self.n_factors)
        )
        self.Q = 0.1 * rng.normal(
            size=(self.n_items, self.n_factors)
        )

        # SVD++ implicit feedback vector
        self.Y = 0.1 * rng.normal(
            size=(self.n_items, self.n_factors)
        )

        self._build_user_history(train_idx_df)
        self._build_item_neighbors(train_idx_df)

        # neighborhood parameter
        # W: explicit residual weight
        # C: implicit neighborhood bias
        self.W = np.zeros(
            (self.n_items, self.neighbors.shape[1])
        )
        self.C = np.zeros(
            (self.n_items, self.neighbors.shape[1])
        )

        train_array = train_idx_df[
            ["user_idx", "item_idx", "rating"]
        ].values

        self.best_state = None
        self.best_val_rmse = np.inf
        self.best_epoch = None

        for epoch in range(self.n_epochs):
            rng.shuffle(train_array)

            squared_errors = []

            for u, i, r in train_array:
                u = int(u)
                i = int(i)
                r = float(r)

                Nu = self.user_items[u]
                sqrt_Nu = math.sqrt(len(Nu)) if len(Nu) > 0 else 1.0

                implicit_sum = np.sum(
                    self.Y[Nu],
                    axis=0
                ) / sqrt_Nu if len(Nu) > 0 else np.zeros(self.n_factors)

                user_vector = self.P[u] + implicit_sum

                baseline_ui = self.mu + self.bu[u] + self.bi[i]
                latent_part = np.dot(self.Q[i], user_vector)

                # neighborhood component
                hist = self.user_history[u]
                neigh_pos = []
                neigh_items = []

                for pos, j in enumerate(self.neighbors[i]):
                    j = int(j)
                    if j in hist and j != i:
                        neigh_pos.append(pos)
                        neigh_items.append(j)

                neighbor_part = 0.0

                if len(neigh_items) > 0:
                    sqrt_m = math.sqrt(len(neigh_items))
                    pos_arr = np.array(neigh_pos)

                    diffs = np.array([
                        hist[j] - (self.mu + self.bu[u] + self.bi[j])
                        for j in neigh_items
                    ])

                    w_vals = self.W[i, pos_arr]
                    c_vals = self.C[i, pos_arr]

                    neighbor_part = np.sum(
                        w_vals * diffs + c_vals
                    ) / sqrt_m
                else:
                    sqrt_m = 1.0
                    pos_arr = None
                    diffs = None

                pred = baseline_ui + latent_part + neighbor_part
                err = r - pred
                squared_errors.append(err ** 2)

                p_old = self.P[u].copy()
                q_old = self.Q[i].copy()

                # bias update
                self.bu[u] += self.lr * (
                    err - self.reg * self.bu[u]
                )
                self.bi[i] += self.lr * (
                    err - self.reg * self.bi[i]
                )

                # latent factor update
                self.P[u] += self.lr * (
                    err * q_old - self.reg * p_old
                )
                self.Q[i] += self.lr * (
                    err * user_vector - self.reg * q_old
                )

                # implicit feedback update
                for j in Nu:
                    self.Y[j] += self.lr * (
                        err * q_old / sqrt_Nu
                        - self.reg * self.Y[j]
                    )

                # neighborhood parameter update
                if len(neigh_items) > 0:
                    self.W[i, pos_arr] += self.lr * (
                        err * diffs / sqrt_m
                        - self.reg * self.W[i, pos_arr]
                    )

                    self.C[i, pos_arr] += self.lr * (
                        err / sqrt_m
                        - self.reg * self.C[i, pos_arr]
                    )

            train_rmse = np.sqrt(np.mean(squared_errors))

            msg = f"Epoch {epoch + 1}/{self.n_epochs} - Train RMSE: {train_rmse:.4f}"

            if val_df is not None:
                val_pred = self.predict(val_df)
                val_rmse = rmse(
                    val_df["rating"].values,
                    val_pred
                )

                if val_rmse < self.best_val_rmse:
                    self.best_val_rmse = val_rmse
                    self.best_epoch = epoch + 1
                    self._save_best_state()
                    msg += f" | Val RMSE: {val_rmse:.4f} *"
                else:
                    msg += f" | Val RMSE: {val_rmse:.4f}"

            print(msg)

        if val_df is not None and self.best_state is not None:
            self._load_best_state()
            print(
                f"Best checkpoint restored: epoch={self.best_epoch}, "
                f"val_rmse={self.best_val_rmse:.4f}"
            )

        return self

    def _save_best_state(self):
        self.best_state = {
            "mu": self.mu,
            "bu": self.bu.copy(),
            "bi": self.bi.copy(),
            "P": self.P.copy(),
            "Q": self.Q.copy(),
            "Y": self.Y.copy(),
            "W": self.W.copy(),
            "C": self.C.copy()
        }

    def _load_best_state(self):
        state = self.best_state

        self.mu = state["mu"]
        self.bu = state["bu"].copy()
        self.bi = state["bi"].copy()
        self.P = state["P"].copy()
        self.Q = state["Q"].copy()
        self.Y = state["Y"].copy()
        self.W = state["W"].copy()
        self.C = state["C"].copy()

    def predict_one(self, user_id, movie_id):
        # cold-start 처리
        if user_id not in self.user_to_idx:
            return self.mu

        u = self.user_to_idx[user_id]

        if movie_id not in self.item_to_idx:
            return clip_rating(self.mu + self.bu[u])

        i = self.item_to_idx[movie_id]

        Nu = self.user_items[u]
        sqrt_Nu = math.sqrt(len(Nu)) if len(Nu) > 0 else 1.0

        implicit_sum = np.sum(
            self.Y[Nu],
            axis=0
        ) / sqrt_Nu if len(Nu) > 0 else np.zeros(self.n_factors)

        user_vector = self.P[u] + implicit_sum

        baseline_ui = self.mu + self.bu[u] + self.bi[i]
        latent_part = np.dot(self.Q[i], user_vector)

        hist = self.user_history[u]

        neigh_pos = []
        neigh_items = []

        for pos, j in enumerate(self.neighbors[i]):
            j = int(j)
            if j in hist and j != i:
                neigh_pos.append(pos)
                neigh_items.append(j)

        neighbor_part = 0.0

        if len(neigh_items) > 0:
            sqrt_m = math.sqrt(len(neigh_items))
            pos_arr = np.array(neigh_pos)

            diffs = np.array([
                hist[j] - (self.mu + self.bu[u] + self.bi[j])
                for j in neigh_items
            ])

            w_vals = self.W[i, pos_arr]
            c_vals = self.C[i, pos_arr]

            neighbor_part = np.sum(
                w_vals * diffs + c_vals
            ) / sqrt_m

        pred = baseline_ui + latent_part + neighbor_part

        return clip_rating(pred)

    def predict(self, df):
        preds = []

        for row in df.itertuples():
            preds.append(
                self.predict_one(
                    row.user_id,
                    row.movie_id
                )
            )

        return np.array(preds)

    def predict_parts(self, df):
        results = []

        for row in df.itertuples():
            user_id = row.user_id
            movie_id = row.movie_id

            if user_id not in self.user_to_idx:
                baseline = self.mu
                latent = 0.0
                neighbor = 0.0
            elif movie_id not in self.item_to_idx:
                u = self.user_to_idx[user_id]
                baseline = self.mu + self.bu[u]
                latent = 0.0
                neighbor = 0.0
            else:
                u = self.user_to_idx[user_id]
                i = self.item_to_idx[movie_id]

                Nu = self.user_items[u]
                sqrt_Nu = math.sqrt(len(Nu)) if len(Nu) > 0 else 1.0

                implicit_sum = np.sum(
                    self.Y[Nu],
                    axis=0
                ) / sqrt_Nu if len(Nu) > 0 else np.zeros(self.n_factors)

                user_vector = self.P[u] + implicit_sum

                baseline = self.mu + self.bu[u] + self.bi[i]
                latent = np.dot(self.Q[i], user_vector)

                hist = self.user_history[u]
                neigh_pos = []
                neigh_items = []

                for pos, j in enumerate(self.neighbors[i]):
                    j = int(j)
                    if j in hist and j != i:
                        neigh_pos.append(pos)
                        neigh_items.append(j)

                neighbor = 0.0

                if len(neigh_items) > 0:
                    sqrt_m = math.sqrt(len(neigh_items))
                    pos_arr = np.array(neigh_pos)

                    diffs = np.array([
                        hist[j] - (self.mu + self.bu[u] + self.bi[j])
                        for j in neigh_items
                    ])

                    neighbor = np.sum(
                        self.W[i, pos_arr] * diffs
                        + self.C[i, pos_arr]
                    ) / sqrt_m

            results.append({
                "baseline": baseline,
                "latent": latent,
                "neighbor": neighbor,
                "integrated": clip_rating(baseline + latent + neighbor)
            })

        return pd.DataFrame(results)

In [ ]:
integrated_model = IntegratedHybridManual(
    n_factors=20,
    k_neighbor=30,
    n_epochs=20,
    lr=0.005,
    reg=0.02,
    random_state=1234
)

integrated_model.fit(
    train_inner,
    val_df=val
)

Epoch 1/20 - Train RMSE: 1.0311 | Val RMSE: 0.9827 *
Epoch 2/20 - Train RMSE: 0.9423 | Val RMSE: 0.9534 *


KeyboardInterrupt: 

In [ ]:
val["integrated_pred"] = integrated_model.predict(val)

integrated_rmse = rmse(
    val["rating"].values,
    val["integrated_pred"].values
)

integrated_mae = mae(
    val["rating"].values,
    val["integrated_pred"].values
)

print("Integrated Hybrid Val RMSE:", integrated_rmse)
print("Integrated Hybrid Val MAE:", integrated_mae)

In [ ]:
parts_df = integrated_model.predict_parts(val)

parts_df.head()

In [ ]:
val_parts = pd.concat(
    [
        val.reset_index(drop=True),
        parts_df.reset_index(drop=True)
    ],
    axis=1
)

val_parts.head()

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Manual Basic SVD",
        "RMSE": rmse_manual_svd,
        "MAE": mae_manual_svd
    },
    {
        "Model": "Manual SVD++",
        "RMSE": rmse_manual_svdpp,
        "MAE": mae_manual_svdpp
    },
    {
        "Model": "Manual Integrated Hybrid",
        "RMSE": integrated_rmse,
        "MAE": integrated_mae
    }
])

comparison_df = comparison_df.sort_values(
    "RMSE"
).reset_index(drop=True)

display(comparison_df)

모델 저장

In [ ]:
import pickle

with open(SAVE_DIR + "integrated_model.pkl", "wb") as f:
    pickle.dump(integrated_model, f)

print("Integrated Hybrid 저장 완료")

Integrated Hybrid 저장 완료


In [ ]:
with open(SAVE_DIR + "integrated_model.pkl", "rb") as f:
    integrated_model = pickle.load(f)

# 앙상블
mean-centered + top-k , Manual Integrated Hybrid

## weighted hybrids

가중치 탐색

In [ ]:
weighted_integrated_results = []

for alpha in np.arange(0, 1.01, 0.05):
    pred = (
        alpha * val["item_cf_topk_pred"]
        +
        (1 - alpha) * val["integrated_pred"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            val["rating"],
            pred
        )
    )

    weighted_integrated_results.append({
        "alpha_item_cf": alpha,
        "alpha_integrated": 1 - alpha,
        "RMSE": rmse
    })

weighted_integrated_df = pd.DataFrame(weighted_integrated_results)

display(
    weighted_integrated_df.sort_values("RMSE")
)

KeyError: 'integrated_pred'

최적 가중치 저장

In [ ]:
best_weighted_integrated = weighted_integrated_df.loc[
    weighted_integrated_df["RMSE"].idxmin()
]

best_alpha_item_cf_integrated = best_weighted_integrated["alpha_item_cf"]

print("Best alpha for Item-CF:", best_alpha_item_cf_integrated)
print("Best alpha for Integrated Hybrid:", 1 - best_alpha_item_cf_integrated)
print("Best Weighted Integrated RMSE:", best_weighted_integrated["RMSE"])

val["weighted_integrated_pred"] = (
    best_alpha_item_cf_integrated * val["item_cf_topk_pred"]
    + (1 - best_alpha_item_cf_integrated) * val["integrated_pred"]
)

Best alpha for Item-CF: 0.35000000000000003
Best alpha for Integrated Hybrid: 0.6499999999999999
Best Weighted Integrated RMSE: 0.909390705052704


기존 모델들과 비교

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

weighted_comparison_df = pd.DataFrame([
    {
        "Model": "Mean-Centered + Top-K Item-CF",
        "RMSE": np.sqrt(mean_squared_error(val["rating"], val["item_cf_topk_pred"])),
        "MAE": mean_absolute_error(val["rating"], val["item_cf_topk_pred"])
    },
    {
        "Model": "Manual Integrated Hybrid",
        "RMSE": integrated_rmse,
        "MAE": integrated_mae
    },
    {
        "Model": "Weighted Ensemble",
        "RMSE": best_weighted_integrated["RMSE"],
        "MAE": mean_absolute_error(
            val["rating"],
            (
                best_alpha_item_cf_integrated * val["item_cf_topk_pred"]
                + (1 - best_alpha_item_cf_integrated) * val["integrated_pred"]
            )
        )
    }
])

display(
    weighted_comparison_df
    .sort_values("RMSE")
    .reset_index(drop=True)
)

,Model,RMSE,MAE
0,Weighted Ensemble,0.909391,0.715052
1,Manual Integrated Hybrid,0.915538,0.720732
2,Mean-Centered + Top-K Item-CF,0.930488,0.728177


저장

In [ ]:
with open(
    SAVE_DIR + "best_alpha_item_cf_integrated.pkl",
    "wb"
) as f:
    pickle.dump(
        best_alpha_item_cf_integrated,
        f
    )

In [ ]:
with open(
    SAVE_DIR + "best_alpha_item_cf_integrated.pkl",
    "rb"
) as f:
    best_alpha_item_cf_integrated = pickle.load(f)

## bagging

Bagging

In [ ]:
bagging_models = []

seeds = [0, 1, 2]

for seed in seeds:

    sampled_train = train_inner.sample(
        frac=1.0,
        replace=True,
        random_state=seed
    )

    model = IntegratedHybridManual(
        n_factors=20,
        k_neighbor=30,
        n_epochs=10,
        lr=0.005,
        reg=0.02,
        random_state=seed
    )

    model.fit(
        sampled_train,
        val_df=val
    )

    bagging_models.append(model)

    print(f"Bagging Model {seed} 완료")

KeyboardInterrupt: 

validationa 계산

In [ ]:
bagging_preds = []

for model in bagging_models:
    preds = model.predict(val)
    bagging_preds.append(preds)

val["integrated_bagging_pred"] = np.mean(
    bagging_preds,
    axis=0
)

print(val[["rating", "integrated_bagging_pred"]].head())

       rating  integrated_bagging_pred
30016       5                 4.629369
51012       3                 3.511088
79374       4                 3.179933
30415       5                 4.312344
69284       3                 3.608286


RMSE 확인

In [ ]:
integrated_bagging_rmse = np.sqrt(
    mean_squared_error(
        val["rating"],
        val["integrated_bagging_pred"]
    )
)

integrated_bagging_mae = mean_absolute_error(
    val["rating"],
    val["integrated_bagging_pred"]
)

print(
    "Integrated Bagging RMSE:",
    integrated_bagging_rmse
)

print(
    "Integrated Bagging MAE:",
    integrated_bagging_mae
)

Integrated Bagging RMSE: 0.9384517543832989
Integrated Bagging MAE: 0.7400622872522862


가중 하이브리드 vs 배깅

Bagging SVD++를 hybrid에 넣기

In [ ]:
bagging_hybrid_results = []

for alpha in np.arange(0, 1.01, 0.05):
    pred = (
        alpha * val["item_cf_topk_pred"] +
        (1 - alpha) * val["svdpp_bagging_pred"]
    )

    rmse = np.sqrt(
        mean_squared_error(val["rating"], pred)
    )

    bagging_hybrid_results.append({
        "alpha_item_cf": alpha,
        "alpha_svdpp_bagging": 1 - alpha,
        "RMSE": rmse
    })

bagging_hybrid_df = pd.DataFrame(bagging_hybrid_results)
bagging_hybrid_df.sort_values("RMSE").head()

,alpha_item_cf,alpha_svdpp_bagging,RMSE
7,0.35,0.65,0.910924
8,0.40,0.60,0.910928
6,0.30,0.70,0.911173
9,0.45,0.55,0.911184
5,0.25,0.75,0.911675


In [ ]:
best_bagging_hybrid = bagging_hybrid_df.loc[
    bagging_hybrid_df["RMSE"].idxmin()
]

print("Best alpha for Item-CF:", best_bagging_hybrid["alpha_item_cf"])
print("Best alpha for SVD++ Bagging:", best_bagging_hybrid["alpha_svdpp_bagging"])
print("Best Bagging Hybrid RMSE:", best_bagging_hybrid["RMSE"])

Best alpha for Item-CF: 0.35000000000000003
Best alpha for SVD++ Bagging: 0.6499999999999999
Best Bagging Hybrid RMSE: 0.9109242483182436


저장

In [ ]:
with open(SAVE_DIR + "bagging_models.pkl", "wb") as f:
    pickle.dump(bagging_models, f)

print("Bagging Models 저장 완료")

Bagging Models 저장 완료


In [ ]:
with open(SAVE_DIR + "bagging_models.pkl", "rb") as f:
    bagging_models = pickle.load(f)

## Boosting-style Sequential Ensemble
예측 오차가 큰 rating 샘플 가중치 증가

In [ ]:
import numpy as np
import pandas as pd
import math

from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error


def clip_rating(pred):
    return np.clip(pred, 1, 5)


def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def calc_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

In [ ]:
class boosting:
    def __init__(
        self,
        n_factors=20,
        k_neighbor=30,
        n_epochs=10,
        lr=0.005,
        reg=0.02,
        random_state=1234
    ):
        self.n_factors = n_factors
        self.k_neighbor = k_neighbor
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def _make_index(self, train_df):
        self.user_ids = train_df["user_id"].unique()
        self.item_ids = train_df["movie_id"].unique()

        self.user_to_idx = {
            user_id: idx for idx, user_id in enumerate(self.user_ids)
        }

        self.item_to_idx = {
            item_id: idx for idx, item_id in enumerate(self.item_ids)
        }

        self.n_users = len(self.user_ids)
        self.n_items = len(self.item_ids)

    def _convert_df(self, df):
        temp = df.copy()

        temp["user_idx"] = temp["user_id"].map(self.user_to_idx)
        temp["item_idx"] = temp["movie_id"].map(self.item_to_idx)

        temp = temp.dropna(subset=["user_idx", "item_idx"])

        temp["user_idx"] = temp["user_idx"].astype(int)
        temp["item_idx"] = temp["item_idx"].astype(int)

        return temp

    def _build_user_history(self, train_idx_df):
        self.user_history = [dict() for _ in range(self.n_users)]

        for row in train_idx_df.itertuples():
            self.user_history[row.user_idx][row.item_idx] = float(row.rating)

        self.user_items = [
            list(history.keys()) for history in self.user_history
        ]

    def _build_item_neighbors(self, train_idx_df):
        rows = train_idx_df["user_idx"].values
        cols = train_idx_df["item_idx"].values
        ratings = train_idx_df["rating"].values - self.mu

        R_sparse = csr_matrix(
            (ratings, (rows, cols)),
            shape=(self.n_users, self.n_items)
        )

        item_norms = np.sqrt(
            np.array(R_sparse.power(2).sum(axis=0)).flatten()
        ) + 1e-9

        sim = (R_sparse.T @ R_sparse).toarray()
        sim = sim / np.outer(item_norms, item_norms)

        np.fill_diagonal(sim, -np.inf)

        top_k = min(self.k_neighbor, self.n_items - 1)

        self.neighbors = np.argpartition(
            -sim,
            top_k,
            axis=1
        )[:, :top_k]

    def _save_best_state(self):
        self.best_state = {
            "mu": self.mu,
            "bu": self.bu.copy(),
            "bi": self.bi.copy(),
            "P": self.P.copy(),
            "Q": self.Q.copy(),
            "Y": self.Y.copy(),
            "W": self.W.copy(),
            "C": self.C.copy()
        }

    def _load_best_state(self):
        state = self.best_state

        self.mu = state["mu"]
        self.bu = state["bu"].copy()
        self.bi = state["bi"].copy()
        self.P = state["P"].copy()
        self.Q = state["Q"].copy()
        self.Y = state["Y"].copy()
        self.W = state["W"].copy()
        self.C = state["C"].copy()

    def fit(self, train_df, val_df=None, sample_weight=None):
        rng = np.random.default_rng(self.random_state)

        train_df = train_df.reset_index(drop=True).copy()

        self._make_index(train_df)

        train_idx_df = self._convert_df(train_df)

        self.mu = train_idx_df["rating"].mean()

        if sample_weight is None:
            train_idx_df["sample_weight"] = 1.0
        else:
            sample_weight = np.asarray(sample_weight)

            if len(sample_weight) != len(train_df):
                raise ValueError(
                    "sample_weight 길이는 train_df 길이와 같아야 합니다."
                )

            train_idx_df["sample_weight"] = sample_weight[
                train_idx_df.index.values
            ]

        self.bu = np.zeros(self.n_users)
        self.bi = np.zeros(self.n_items)

        self.P = 0.1 * rng.normal(
            size=(self.n_users, self.n_factors)
        )

        self.Q = 0.1 * rng.normal(
            size=(self.n_items, self.n_factors)
        )

        self.Y = 0.1 * rng.normal(
            size=(self.n_items, self.n_factors)
        )

        self._build_user_history(train_idx_df)
        self._build_item_neighbors(train_idx_df)

        self.W = np.zeros(
            (self.n_items, self.neighbors.shape[1])
        )

        self.C = np.zeros(
            (self.n_items, self.neighbors.shape[1])
        )

        train_array = train_idx_df[
            ["user_idx", "item_idx", "rating", "sample_weight"]
        ].values

        self.best_state = None
        self.best_val_rmse = np.inf
        self.best_epoch = None

        for epoch in range(self.n_epochs):
            rng.shuffle(train_array)

            squared_errors = []

            for u, i, r, sw in train_array:
                u = int(u)
                i = int(i)
                r = float(r)
                sw = float(sw)

                Nu = self.user_items[u]
                sqrt_Nu = math.sqrt(len(Nu)) if len(Nu) > 0 else 1.0

                if len(Nu) > 0:
                    implicit_sum = np.sum(
                        self.Y[Nu],
                        axis=0
                    ) / sqrt_Nu
                else:
                    implicit_sum = np.zeros(self.n_factors)

                user_vector = self.P[u] + implicit_sum

                baseline_ui = self.mu + self.bu[u] + self.bi[i]
                latent_part = np.dot(self.Q[i], user_vector)

                hist = self.user_history[u]

                neigh_pos = []
                neigh_items = []

                for pos, j in enumerate(self.neighbors[i]):
                    j = int(j)

                    if j in hist and j != i:
                        neigh_pos.append(pos)
                        neigh_items.append(j)

                neighbor_part = 0.0

                if len(neigh_items) > 0:
                    sqrt_m = math.sqrt(len(neigh_items))
                    pos_arr = np.array(neigh_pos)

                    diffs = np.array([
                        hist[j] - (
                            self.mu + self.bu[u] + self.bi[j]
                        )
                        for j in neigh_items
                    ])

                    w_vals = self.W[i, pos_arr]
                    c_vals = self.C[i, pos_arr]

                    neighbor_part = np.sum(
                        w_vals * diffs + c_vals
                    ) / sqrt_m
                else:
                    sqrt_m = 1.0
                    pos_arr = None
                    diffs = None

                pred = baseline_ui + latent_part + neighbor_part

                err = r - pred
                weighted_err = sw * err

                squared_errors.append(err ** 2)

                p_old = self.P[u].copy()
                q_old = self.Q[i].copy()

                self.bu[u] += self.lr * (
                    weighted_err - self.reg * self.bu[u]
                )

                self.bi[i] += self.lr * (
                    weighted_err - self.reg * self.bi[i]
                )

                self.P[u] += self.lr * (
                    weighted_err * q_old - self.reg * p_old
                )

                self.Q[i] += self.lr * (
                    weighted_err * user_vector - self.reg * q_old
                )

                for j in Nu:
                    self.Y[j] += self.lr * (
                        weighted_err * q_old / sqrt_Nu
                        - self.reg * self.Y[j]
                    )

                if len(neigh_items) > 0:
                    self.W[i, pos_arr] += self.lr * (
                        weighted_err * diffs / sqrt_m
                        - self.reg * self.W[i, pos_arr]
                    )

                    self.C[i, pos_arr] += self.lr * (
                        weighted_err / sqrt_m
                        - self.reg * self.C[i, pos_arr]
                    )

            train_rmse = np.sqrt(np.mean(squared_errors))

            msg = (
                f"Epoch {epoch + 1}/{self.n_epochs} "
                f"- Train RMSE: {train_rmse:.4f}"
            )

            if val_df is not None:
                val_pred = self.predict(val_df)

                val_rmse = calc_rmse(
                    val_df["rating"].values,
                    val_pred
                )

                if val_rmse < self.best_val_rmse:
                    self.best_val_rmse = val_rmse
                    self.best_epoch = epoch + 1
                    self._save_best_state()
                    msg += f" | Val RMSE: {val_rmse:.4f} *"
                else:
                    msg += f" | Val RMSE: {val_rmse:.4f}"

            print(msg)

        if val_df is not None and self.best_state is not None:
            self._load_best_state()

            print(
                f"Best checkpoint restored: "
                f"epoch={self.best_epoch}, "
                f"val_rmse={self.best_val_rmse:.4f}"
            )

        return self

    def predict_one(self, user_id, movie_id):
        if user_id not in self.user_to_idx:
            return self.mu

        u = self.user_to_idx[user_id]

        if movie_id not in self.item_to_idx:
            return clip_rating(self.mu + self.bu[u])

        i = self.item_to_idx[movie_id]

        Nu = self.user_items[u]
        sqrt_Nu = math.sqrt(len(Nu)) if len(Nu) > 0 else 1.0

        if len(Nu) > 0:
            implicit_sum = np.sum(
                self.Y[Nu],
                axis=0
            ) / sqrt_Nu
        else:
            implicit_sum = np.zeros(self.n_factors)

        user_vector = self.P[u] + implicit_sum

        baseline_ui = self.mu + self.bu[u] + self.bi[i]
        latent_part = np.dot(self.Q[i], user_vector)

        hist = self.user_history[u]

        neigh_pos = []
        neigh_items = []

        for pos, j in enumerate(self.neighbors[i]):
            j = int(j)

            if j in hist and j != i:
                neigh_pos.append(pos)
                neigh_items.append(j)

        neighbor_part = 0.0

        if len(neigh_items) > 0:
            sqrt_m = math.sqrt(len(neigh_items))
            pos_arr = np.array(neigh_pos)

            diffs = np.array([
                hist[j] - (
                    self.mu + self.bu[u] + self.bi[j]
                )
                for j in neigh_items
            ])

            neighbor_part = np.sum(
                self.W[i, pos_arr] * diffs
                + self.C[i, pos_arr]
            ) / sqrt_m

        pred = baseline_ui + latent_part + neighbor_part

        return clip_rating(pred)

    def predict(self, df):
        preds = []

        for row in df.itertuples():
            pred = self.predict_one(
                row.user_id,
                row.movie_id
            )
            preds.append(pred)

        return np.array(preds)

학습

In [ ]:
boosting_models = []
model_weights = []

n_estimators = 3
boost_lr = 0.5

train_boost_df = train_inner.reset_index(drop=True).copy()

sample_weight = np.ones(len(train_boost_df))
sample_weight = sample_weight / sample_weight.mean()

In [ ]:
for m in range(n_estimators):

    print(f"\n===== Boosting Round {m + 1}/{n_estimators} =====")

    model = boosting(
        n_factors=20,
        k_neighbor=30,
        n_epochs=10,
        lr=0.005,
        reg=0.02,
        random_state=1234 + m
    )

    model.fit(
        train_boost_df,
        val_df=val,
        sample_weight=sample_weight
    )

    train_pred = model.predict(train_boost_df)
    train_true = train_boost_df["rating"].values

    abs_error = np.abs(train_true - train_pred)

    model_rmse = np.sqrt(
        mean_squared_error(
            train_true,
            train_pred
        )
    )

    model_alpha = 1 / (model_rmse + 1e-8)

    boosting_models.append(model)
    model_weights.append(model_alpha)

    sample_weight = sample_weight * np.exp(
        boost_lr * abs_error
    )

    sample_weight = sample_weight / sample_weight.mean()

    print("Train RMSE:", model_rmse)
    print("Model alpha:", model_alpha)


===== Boosting Round 1/3 =====
Epoch 1/10 - Train RMSE: 1.0311 | Val RMSE: 0.9827 *
Epoch 2/10 - Train RMSE: 0.9423 | Val RMSE: 0.9534 *
Epoch 3/10 - Train RMSE: 0.9053 | Val RMSE: 0.9402 *
Epoch 4/10 - Train RMSE: 0.8807 | Val RMSE: 0.9326 *
Epoch 5/10 - Train RMSE: 0.8619 | Val RMSE: 0.9275 *
Epoch 6/10 - Train RMSE: 0.8463 | Val RMSE: 0.9239 *
Epoch 7/10 - Train RMSE: 0.8328 | Val RMSE: 0.9215 *
Epoch 8/10 - Train RMSE: 0.8208 | Val RMSE: 0.9194 *
Epoch 9/10 - Train RMSE: 0.8101 | Val RMSE: 0.9184 *
Epoch 10/10 - Train RMSE: 0.8001 | Val RMSE: 0.9175 *
Best checkpoint restored: epoch=10, val_rmse=0.9175
Train RMSE: 0.7877730938227672
Model alpha: 1.269401043457025

===== Boosting Round 2/3 =====
Epoch 1/10 - Train RMSE: 1.0401 | Val RMSE: 0.9898 *
Epoch 2/10 - Train RMSE: 0.9465 | Val RMSE: 0.9584 *
Epoch 3/10 - Train RMSE: 0.9045 | Val RMSE: 0.9440 *
Epoch 4/10 - Train RMSE: 0.8756 | Val RMSE: 0.9362 *
Epoch 5/10 - Train RMSE: 0.8533 | Val RMSE: 0.9305 *
Epoch 6/10 - Train RMSE: 0

In [ ]:
def predict_boosting(models, model_weights, df):
    """
    Boosting Ensemble Prediction
    """

    model_weights = np.array(model_weights)

    # 정규화
    model_weights = (
        model_weights
        / model_weights.sum()
    )

    all_preds = []

    for model in models:
        preds = model.predict(df)
        all_preds.append(preds)

    all_preds = np.array(all_preds)

    final_pred = np.average(
        all_preds,
        axis=0,
        weights=model_weights
    )

    return np.clip(
        final_pred,
        1,
        5
    )

In [ ]:
test["boosting_pred"] = predict_boosting(
    boosting_models,
    model_weights,
    test
)

# 모델 저장

Item-CF에 필요한 객체 저장

In [ ]:
item_cf_objects = {
    "train_matrix": train_matrix,
    "item_similarity_centered_df": item_similarity_centered_df,
    "user_means": user_means,
    "global_mean": global_mean,
    "best_k": best_k
}

with open(SAVE_DIR + "item_cf_topk_objects.pkl", "wb") as f:
    pickle.dump(item_cf_objects, f)

NameError: name 'item_similarity_centered_df' is not defined

In [ ]:
all_models = {
    "integrated_model": integrated_model,
    "bagging_models": bagging_models,
    "boosting_models": boosting_models,
    "boosting_weights": model_weights,
    "best_alpha": best_alpha_item_cf_integrated
}

with open(
    SAVE_DIR + "all_recommender_models.pkl",
    "wb"
) as f:
    pickle.dump(all_models, f)

print("전체 모델 저장 완료")

전체 모델 저장 완료


In [ ]:
import pickle

with open(
    SAVE_DIR + "all_recommender_models.pkl",
    "rb"
) as f:
    all_models = pickle.load(f)

integrated_model = all_models["integrated_model"]
bagging_models = all_models["bagging_models"]
boosting_models = all_models["boosting_models"]
model_weights = all_models["boosting_weights"]
best_alpha_item_cf_integrated = all_models["best_alpha"]

print("전체 모델 불러오기 완료")

AttributeError: Can't get attribute 'boosting' on <module '__main__'>

## test 예측 결과 CSV 저장

In [ ]:
test.to_csv(
    SAVE_DIR + "test_predictions_all_models.csv",
    index=False
)

print("저장 완료")

저장 완료


In [ ]:
print(test.columns)
print(test.shape)

Index(['user_id', 'movie_id', 'rating', 'integrated_pred', 'item_cf_topk_pred',
       'weighted_integrated_pred', 'integrated_bagging_pred', 'boosting_pred'],
      dtype='object')
(9430, 8)


In [ ]:
test_evaluation_df.to_csv(
    SAVE_DIR + "test_model_comparison.csv",
    index=False
)

print("평가결과 저장 완료")

평가결과 저장 완료


파일 불러오기

In [ ]:
import pandas as pd

test = pd.read_csv(SAVE_DIR + "test_with_predictions.csv")
val = pd.read_csv(SAVE_DIR + "val_with_predictions.csv")

# 카탈로그 커버리지 단점 해결 모델

정확도 성능이 좋지만 카탈로그 커버리지 성능이 낮은 weighted hybrids 모델에 대해서 예측 시 아이템 인기도가 낮은 영화에게 가중치를 주는 방식으로 계산해본다.

롱 테일 (비인기 영화) 점수 만들기 : 영화별 평가 횟수를 기준으로 인기도를 계산한다

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

1. 저장해둔 Weighted Ensemble 최적 alpha 불러오기

In [ ]:
with open(SAVE_DIR + "best_alpha_item_cf_integrated.pkl", "rb") as f:
    best_alpha_item_cf_integrated = pickle.load(f)

alpha = best_alpha_item_cf_integrated

print("Best alpha for Item-CF:", alpha)
print("Best alpha for Integrated Hybrid:", 1 - alpha)

Best alpha for Item-CF: 0.35000000000000003
Best alpha for Integrated Hybrid: 0.6499999999999999


저장된 예측 결과 불러오기

In [ ]:
test = pd.read_csv(SAVE_DIR + "test_with_predictions.csv")
val = pd.read_csv(SAVE_DIR + "val_with_predictions.csv")

롱테일 보너스 계산

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

test.columns = test.columns.str.strip()
val.columns = val.columns.str.strip()

# 롱테일 보너스 추가
for df in [test, val]:
    df["longtail_bonus"] = df["movie_id"].map(longtail_bonus_dict)
    df["longtail_bonus"] = df["longtail_bonus"].fillna(1.0)

lambda 탐색: val 기준

In [ ]:
lambda_list = [0, 0.02, 0.05, 0.10, 0.15, 0.20]
results = []

for lam in lambda_list:
    pred = (
        val["ensemble_pred"]
        + lam * val["longtail_bonus"]
    ).clip(1, 5)

    rmse = np.sqrt(mean_squared_error(val["rating"], pred))
    mae = mean_absolute_error(val["rating"], pred)

    results.append({
        "lambda": lam,
        "Val_RMSE": rmse,
        "Val_MAE": mae
    })

val_longtail_results = pd.DataFrame(results)
val_longtail_results

,lambda,Val_RMSE,Val_MAE
0,0.00,0.930432,0.729681
1,0.02,0.932164,0.730286
2,0.05,0.935192,0.731552
3,0.10,0.941367,0.734677
4,0.15,0.948920,0.739024
5,0.20,0.957813,0.744532


가장 RMSE 낮은 lambda 선택

In [ ]:
best_lambda = val_longtail_results.loc[
    val_longtail_results["Val_RMSE"].idxmin(),
    "lambda"
]

print("Best lambda:", best_lambda)

Best lambda: 0.0


롤테일을 반영하지 않을 때 가장 정확도가 높은 결과가 나타남

2. 가중치에서 0 을 제외하고 재탐색

In [ ]:
# lambda 탐색: 0 제외
lambda_list = [0.02, 0.05, 0.10, 0.15, 0.20]

results = []

for lam in lambda_list:
    pred = (
        val["ensemble_pred"]
        + lam * val["longtail_bonus"]
    ).clip(1, 5)

    rmse = np.sqrt(mean_squared_error(val["rating"], pred))
    mae = mean_absolute_error(val["rating"], pred)

    results.append({
        "lambda": lam,
        "Val_RMSE": rmse,
        "Val_MAE": mae
    })

val_longtail_results = pd.DataFrame(results)

best_lambda = val_longtail_results.loc[
    val_longtail_results["Val_RMSE"].idxmin(),
    "lambda"
]

print("Best non-zero lambda:", best_lambda)
val_longtail_results

Best non-zero lambda: 0.02


,lambda,Val_RMSE,Val_MAE
0,0.02,0.932164,0.730286
1,0.05,0.935192,0.731552
2,0.10,0.941367,0.734677
3,0.15,0.948920,0.739024
4,0.20,0.957813,0.744532


In [ ]:
best_lambda = val_longtail_results.loc[
    val_longtail_results["Val_RMSE"].idxmin(),
    "lambda"
]

print("Best lambda:", best_lambda)

Best lambda: 0.02


test 적용

In [ ]:
test["longtail_weighted_pred"] = (
    test["weighted_hybrid_pred"]
    + best_lambda * test["longtail_bonus"]
).clip(1, 5)

test_rmse = np.sqrt(
    mean_squared_error(test["rating"], test["longtail_weighted_pred"])
)

test_mae = mean_absolute_error(
    test["rating"],
    test["longtail_weighted_pred"]
)

print("Long-tail Weighted Ensemble Test RMSE:", test_rmse)
print("Long-tail Weighted Ensemble Test MAE:", test_mae)

Long-tail Weighted Ensemble Test RMSE: 0.9425218789507637
Long-tail Weighted Ensemble Test MAE: 0.7449757145712333


카탈로그 커버리지 재계산

In [ ]:
import numpy as np
import pandas as pd

# 전체 영화 목록
all_movies = sorted(
    pd.concat([
        train[["movie_id"]],
        test[["movie_id"]]
    ])["movie_id"].unique()
)

total_items = len(all_movies)

# 사용자별 이미 본 영화
user_seen_items = (
    train
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

test_users = sorted(test["user_id"].unique())

print("users:", len(test_users))
print("items:", total_items)

users: 943
items: 1682


In [ ]:
def recommend_longtail_topk_for_user(
    user_id,
    k,
    best_lambda
):
    seen_items = user_seen_items.get(user_id, set())
    candidate_items = [
        movie_id for movie_id in all_movies
        if movie_id not in seen_items
    ]

    preds = []

    for movie_id in candidate_items:
        # 기존 모델 예측값
        item_cf_pred = predict_item_cf_topk(user_id, movie_id)
        svdpp_pred = svdpp_model.predict(user_id, movie_id).est

        # 기존 Weighted Hybrid 예측
        weighted_pred = (
            best_alpha_item_cf_integrated * item_cf_pred
            + (1 - best_alpha_item_cf_integrated) * svdpp_pred
        )

        # 롱테일 보너스
        bonus = longtail_bonus_dict.get(movie_id, 1.0)

        # 최종 점수
        final_pred = weighted_pred + best_lambda * bonus
        final_pred = np.clip(final_pred, 1, 5)

        preds.append((movie_id, final_pred))

    topk_items = [
        movie_id for movie_id, score in
        sorted(preds, key=lambda x: x[1], reverse=True)[:k]
    ]

    return topk_items

In [ ]:
def catalog_coverage_longtail(k, best_lambda):
    recommended_items = set()

    for user_id in test_users:
        topk_items = recommend_longtail_topk_for_user(
            user_id=user_id,
            k=k,
            best_lambda=best_lambda
        )
        recommended_items.update(topk_items)

    coverage = len(recommended_items) / total_items
    return coverage

In [ ]:
def catalog_coverage_from_recommendations(
    recommendations,
    all_movies,
    k
):
    recommended_items = set()

    for user_id, rec_list in recommendations.items():
        topk_items = rec_list[:k]
        recommended_items.update(topk_items)

    return len(recommended_items) / len(all_movies)

In [ ]:
# Weighted Ensemble 추천 결과만 추출
weighted_recs = full_catalog_recommendations["Weighted Ensemble"]

# 롱테일 재정렬 함수
def rerank_with_longtail_bonus(recommendations, longtail_bonus_dict, lambda_):
    reranked_recommendations = {}

    for user_id, rec_list in recommendations.items():
        new_rec_list = []

        for movie_id, score in rec_list:
            bonus = longtail_bonus_dict.get(movie_id, 1.0)
            new_score = score + lambda_ * bonus
            new_rec_list.append((movie_id, new_score))

        # 롱테일 보너스 반영 후 재정렬
        new_rec_list = sorted(
            new_rec_list,
            key=lambda x: x[1],
            reverse=True
        )

        reranked_recommendations[user_id] = new_rec_list

    return reranked_recommendations

In [ ]:
# lambda 적용
lambda_ = best_lambda   # 예: 0.02 또는 0.05

longtail_weighted_recs = rerank_with_longtail_bonus(
    recommendations=weighted_recs,
    longtail_bonus_dict=longtail_bonus_dict,
    lambda_=lambda_
)

In [ ]:
def catalog_coverage_from_scored_recs(recommendations, total_items, k):
    recommended_items = set()

    for user_id, rec_list in recommendations.items():
        topk_items = [
            movie_id for movie_id, score in rec_list[:k]
        ]
        recommended_items.update(topk_items)

    return len(recommended_items) / total_items

In [ ]:
coverage_results = []

for model_name, recs in [
    ("Weighted Ensemble", weighted_recs),
    ("Long-tail Weighted Ensemble", longtail_weighted_recs)
]:
    for k in [10, 20, 30]:
        cov = catalog_coverage_from_scored_recs(
            recommendations=recs,
            total_items=total_items,
            k=k
        )

        coverage_results.append({
            "Model": model_name,
            "K": k,
            "Catalog Coverage": cov

        })

coverage_compare_df = pd.DataFrame(coverage_results)
coverage_compare_df

,Model,K,Catalog Coverage
0,Weighted Ensemble,10,0.328181
1,Weighted Ensemble,20,0.422711
2,Weighted Ensemble,30,0.476813
3,Long-tail Weighted Ensemble,10,0.331153
4,Long-tail Weighted Ensemble,20,0.422117
5,Long-tail Weighted Ensemble,30,0.476813


큰 변화가 없어 모든 람다값에 대해서 재계산

In [ ]:
lambda_list = [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

all_lambda_coverage_results = []

for lam in lambda_list:
    longtail_recs = rerank_with_longtail_bonus(
        recommendations=weighted_recs,
        longtail_bonus_dict=longtail_bonus_dict,
        lambda_=lam
    )

    for k in [10, 20, 30]:
        cov = catalog_coverage_from_scored_recs(
            recommendations=longtail_recs,
            total_items=total_items,
            k=k
        )

        all_lambda_coverage_results.append({
            "lambda": lam,
            "K": k,
            "Catalog Coverage": cov,
            "Catalog Coverage Percent": cov * 100
        })

lambda_coverage_df = pd.DataFrame(all_lambda_coverage_results)

lambda_coverage_pivot = lambda_coverage_df.pivot(
    index="lambda",
    columns="K",
    values="Catalog Coverage"
)

lambda_coverage_pivot

K,10,20,30
lambda,,,
0.00,0.328181,0.422711,0.476813
0.02,0.331153,0.422117,0.476813
0.05,0.334126,0.427467,0.476813
0.10,0.343044,0.436980,0.476813
0.15,0.343639,0.442925,0.476813
0.20,0.347206,0.447087,0.476813
0.30,0.350773,0.450654,0.476813
0.50,0.361474,0.448276,0.476813


람다 값에 따른 RMSE, MAE 비교

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

lambda_list = [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

rmse_mae_results = []

for lam in lambda_list:

    pred = (
        test["weighted_hybrid_pred"]
        + lam * test["longtail_bonus"]
    ).clip(1, 5)

    rmse = np.sqrt(
        mean_squared_error(
            test["rating"],
            pred
        )
    )

    mae = mean_absolute_error(
        test["rating"],
        pred
    )

    rmse_mae_results.append({
        "lambda": lam,
        "RMSE": rmse,
        "MAE": mae
    })

rmse_mae_df = pd.DataFrame(rmse_mae_results)

rmse_mae_df

,lambda,RMSE,MAE
0,0.00,0.941903,0.745354
1,0.02,0.942522,0.744976
2,0.05,0.943846,0.744675
3,0.10,0.947106,0.744952
4,0.15,0.951663,0.746265
5,0.20,0.957500,0.748471
6,0.30,0.972900,0.755737
7,0.50,1.017577,0.782160


In [ ]:
coverage10 = (
    lambda_coverage_df[
        lambda_coverage_df["K"] == 10
    ][["lambda","Catalog Coverage"]]
    .rename(
        columns={
            "Catalog Coverage":"Coverage@10"
        }
    )
)

coverage20 = (
    lambda_coverage_df[
        lambda_coverage_df["K"] == 20
    ][["lambda","Catalog Coverage"]]
    .rename(
        columns={
            "Catalog Coverage":"Coverage@20"
        }
    )
)

coverage30 = (
    lambda_coverage_df[
        lambda_coverage_df["K"] == 30
    ][["lambda","Catalog Coverage"]]
    .rename(
        columns={
            "Catalog Coverage":"Coverage@30"
        }
    )
)

comparison_df = (
    rmse_mae_df
    .merge(coverage10,on="lambda")
    .merge(coverage20,on="lambda")
    .merge(coverage30,on="lambda")
)

comparison_df

,lambda,RMSE,MAE,Coverage@10,Coverage@20,Coverage@30
0,0.00,0.941903,0.745354,0.328181,0.422711,0.476813
1,0.02,0.942522,0.744976,0.331153,0.422117,0.476813
2,0.05,0.943846,0.744675,0.334126,0.427467,0.476813
3,0.10,0.947106,0.744952,0.343044,0.436980,0.476813
4,0.15,0.951663,0.746265,0.343639,0.442925,0.476813
5,0.20,0.957500,0.748471,0.347206,0.447087,0.476813
6,0.30,0.972900,0.755737,0.350773,0.450654,0.476813
7,0.50,1.017577,0.782160,0.361474,0.448276,0.476813
